In [ ]:
"""Detect and scrub Wikipedia image-markup fragments left by a broken
wikitext extractor.
 
The delimiters ([[ ]] : |) are already gone from the corpus, so the original
structure is unrecoverable -- this is glued-token repair, not parsing, and it
is lossy by construction. Run diagnose() first: the density distribution tells
you whether to scrub, to drop documents, or to go find a better-extracted
copy of the corpus.
"""
 
import re
from collections import Counter
 
_L    = r"[^\W\d_]"          # any Unicode letter
_EXT  = r"(?:png|jpe?g|gif|svg|webp|tiff?|ogg|ogv|oga|webm|djvu|pdf)"
_SIZE = r"\d+(?:\s?[xх×]\s?\d+)?(?:пкс|px)"
_KW   = (r"(?:міні|мініатюра|thumb|thumbnail|праворуч|ліворуч|справа|зліва"
         r"|right|left|center|центр|межа|border|безрамки|frameless|рамка"
         r"|frame|верх|низ|top|bottom|upright|альт|alt|посилання|link"
         r"|клас|class)")
_LEAD = r"(?:Файл|Зображення|File|Image|Медіа|Media)"
 
# An anchor is something that is essentially never legitimate prose:
#   - a pixel size spec
#   - a file extension glued to a letter (bare or dotted)
#   - a namespace lead glued to a letter on either side
# Trailing markup keywords are absorbed only when they follow an anchor,
# so standalone "межа" / "територія" / "файл" in real sentences survive.
_ANCHOR = (rf"(?:{_SIZE}"
           rf"|(?<={_L})\.?{_EXT}(?![a-zA-Z])"
           rf"|(?<={_L}){_LEAD}|{_LEAD}(?={_L}))")
 
ARTIFACT = re.compile(rf"{_ANCHOR}(?:{_KW})*", re.IGNORECASE)
 
# Cyrillic camelCase: a cheap second signal for the same extraction failure,
# and one the mixed-script counter structurally cannot see.
CYR_GLUE = re.compile(r"[а-щьюяїієґ][А-ЩЬЮЯЇІЄҐ]")
 
def scrub(s: str) -> str:
    return re.sub(r" {2,}", " ", ARTIFACT.sub(" ", s)).strip()
 
 
def diagnose(docs, limit=50_000):
    """Per-document artifact density"""
    buckets = Counter()
    samples, examples = [], Counter()
    for i, doc in enumerate(docs):
        if i >= limit:
            break
        hits = ARTIFACT.findall(doc)
        glue = len(CYR_GLUE.findall(doc))
        examples.update(m if isinstance(m, str) else m[0] for m in hits)
        density = (sum(len(h) for h in hits) / len(doc)) if doc else 0.0
        if not hits and not glue:
            buckets["clean"] += 1
        elif density < 0.005:
            buckets["<0.5%"] += 1
        elif density < 0.02:
            buckets["0.5-2%"] += 1
        elif density < 0.05:
            buckets["2-5%"] += 1
        else:
            buckets[">5%"] += 1
            if len(samples) < 5:
                samples.append(doc[:600])
 
    total = sum(buckets.values())
    for k in ["clean", "<0.5%", "0.5-2%", "2-5%", ">5%"]:
        n = buckets[k]
        print(f"{k:>8}  {n:>8,}  {n/total*100:5.2f}%")
    print("\ntop artifact strings:")
    for s, n in examples.most_common(25):
        print(f"  {n:>7,}  {s!r}")
    print("\n--- worst documents ---")
    for s in samples:
        print(s, "\n" + "-" * 60)
    return buckets


In [4]:
import unicodedata
import re

# --------------------------------------------------------------------------
# stress marks
# --------------------------------------------------------------------------
# Ukrainian marks stress with an acute that is not part of standard
# orthography. Two complications:
#   1. й (U+0439) and ї (U+0457) decompose under NFD into base + combining
#      mark, so the usual "strip everything with a combining class" recipe
#      silently destroys them. Hence the final NFC and the narrow mark class.
#   2. Stress is sometimes typed as a precomposed Latin vowel (ó = U+00F3)
#      because it is reachable on a keyboard. NFD exposes the Latin base,
#      which we map to Cyrillic ONLY when it carries a combining mark and
#      sits in an otherwise-Cyrillic word.
 
_LAT_VOWEL = {
    "a": "а", "e": "е", "i": "і", "o": "о", "y": "у",
    "A": "А", "E": "Е", "I": "І", "O": "О", "Y": "У",
}
 
_WORD         = re.compile(r"[\w\u0300-\u036F]+")
_HAS_CYR      = re.compile(r"[\u0400-\u04FF]")
_LAT_STRESSED = re.compile(r"([aeioyAEIOY])(?=[\u0300\u0301])")
_STRESS       = re.compile(r"([\u0400-\u04FF\u0500-\u052F])[\u0300\u0301]+")
 
 
def _fix_word(m: "re.Match") -> str:
    w = m.group(0)
    if not _HAS_CYR.search(w):
        return w  # pure Latin: café, iPhone -- leave alone
    # Lookahead keeps the mark in place; _STRESS removes it on the next pass.
    return _LAT_STRESSED.sub(lambda k: _LAT_VOWEL[k.group(1)], w)
 
 
def strip_stress(s: str) -> str:
    s = unicodedata.normalize("NFD", s)   # splits ó, and precomposed ѐ/ѝ
    s = _WORD.sub(_fix_word, s)           # Latin stressed vowel -> Cyrillic
    s = _STRESS.sub(r"\1", s)             # drop acute/grave on Cyrillic bases
    return unicodedata.normalize("NFC", s)  # rebuild й, ї, ё, café
 
 
# --------------------------------------------------------------------------
# apostrophes
# --------------------------------------------------------------------------
# The apostrophe is orthographic in Ukrainian (п'ять, об'єкт) and appears as
# at least four code points in the wild. Each variant is a distinct BPE token.
 
_APOS = {ord(c): "'" for c in "\u2019\u2018\u02BC\u02B9\u0060\u00B4"}
 
 
# --------------------------------------------------------------------------
# whitespace and invisibles
# --------------------------------------------------------------------------
 
_DELETE = re.compile(                     # controls, soft hyphen, zero-width
    r"[\u0000-\u0008\u000E-\u001F\u007F-\u009F"
    r"\u00AD"
    r"\u200B-\u200F\u202A-\u202E\u2060-\u2064\u206A-\u206F\uFEFF]"
)
_NEWLINE = re.compile(r"\r\n|[\r\v\f\u2028\u2029]")
_SPACE   = re.compile(r"[\t\u00A0\u1680\u2000-\u200A\u202F\u205F\u3000]")
_RUN     = re.compile(r" {2,}")
_TRIM    = re.compile(r"^ +| +$", re.M)
_BLANKS  = re.compile(r"\n{3,}")
 
 
def clean_ws(s: str) -> str:
    s = _DELETE.sub("", s)
    s = _NEWLINE.sub("\n", s)
    s = _SPACE.sub(" ", s)
    s = _RUN.sub(" ", s)
    s = _TRIM.sub("", s)      # must precede _BLANKS: a space-only line
    s = _BLANKS.sub("\n\n", s)  # blocks the \n{3,} match otherwise
    return s.strip()
 
 
# --------------------------------------------------------------------------
 
def clean_ua_str(s: str) -> str:
    """strip_stress ends in NFC; _APOS and clean_ws only touch ASCII and
    invisibles, so the output is still NFC."""
    return clean_ws(strip_stress(scrub(s)).translate(_APOS))

# --------------------------------------------------------------------------
# tests
# --------------------------------------------------------------------------
 
CASES = [
    # stress removal
    ("Со́нячна систе́ма",      "Сонячна система"),
    ("В\u00F3день",           "Водень"),   # precomposed Latin ó
    ("В\u043E\u0301день",     "Водень"),   # Cyrillic о + combining acute
    ("\u0450 \u045D",         "е и"),      # precomposed grave ѐ ѝ
 
    # letters that must survive: these decompose under NFD
    ("Київ",                  "Київ"),     # ї = і + diaeresis
    ("Йосип",                 "Йосип"),    # й = и + breve
    ("ёлка",                  "ёлка"),     # diaeresis is not stress
    ("ґанок",                 "ґанок"),    # distinct letter, not г
    ("Flёur",                 "Flёur"),    # mixed script, no stress mark
    ("і\u0301\u0308жак",      "їжак"),     # adversarial mark order
 
    # lost word boundaries from the corpus -- must NOT be "repaired"
    ("рокуRHL",               "рокуRHL"),
    ("Anczewskiбурмистр",     "Anczewskiбурмистр"),
    ("АнчевськийMartinus",    "АнчевськийMartinus"),
    ("Al2О3",                 "Al2О3"),
    ("яhttps",                "яhttps"),
    ("Петербург150px",        "Петербург"),
    ("café",                  "café"),
    ("iPhone",                "iPhone"),
 
    # apostrophes
    ("п\u2019ять",            "п'ять"),
    ("здоров\u02BCя",         "здоров'я"),
 
    # whitespace
    ("а  б",                  "а б"),
    ("а\u00A0б",              "а б"),      # NBSP
    ("м'я\u00ADкий",          "м'який"),   # soft hyphen
    ("\uFEFFтекст",           "текст"),    # BOM
    ("рядок   \n\n\n\n  рядок", "рядок\n\nрядок"),
    ("а \n \n б",             "а\n\nб"),   # space-only line
    ("а\r\nб",                "а\nб"),
]
 
 
def _run_tests() -> None:
    failed = 0
    for src, want in CASES:
        got = clean_ua_str(src)
        if got != want:
            failed += 1
            print(f"FAIL {src!r}\n  got  {got!r}\n  want {want!r}")
    for src, _ in CASES:
        out = clean_ua_str(src)
        assert out == unicodedata.normalize("NFC", out), f"not NFC: {src!r}"
    print(f"{len(CASES) - failed}/{len(CASES)} passed")
 
 
if __name__ == "__main__":
    _run_tests()

27/27 passed


In [5]:
#Helpers
import glob, pyarrow.parquet as pq, pyarrow.compute as pc

def get_dataset_stats(pattern):
    rows = chars = nbytes = 0
    nfiles = 0
    for path in sorted(glob.glob(pattern)):
        nfiles += 1
        for b in pq.ParquetFile(path).iter_batches(batch_size=10_000, columns=["text"]):
            col = b.column("text")
            rows   += len(col)
            chars  += pc.sum(pc.utf8_length(col)).as_py() or 0
            nbytes += pc.sum(pc.binary_length(col)).as_py() or 0
           

    print(f"Pattern: {pattern} [{nfiles} files]")
    print(f"{rows:,} docs | {chars:,} chars | {nbytes/chars:.2f} bytes/char")

In [4]:
# Test processing

import glob
import pyarrow.parquet as pq
from collections import Counter


def documents(pattern):
    """Yield one document at a time, as a Python str."""
    for path in sorted(glob.glob(pattern)):
        pf = pq.ParquetFile(path)
        for batch in pf.iter_batches(batch_size=1000, columns=["text"]):
            for doc in batch.column("text").to_pylist():
                if doc:                    # skip nulls / empty rows
                    yield doc

docs = []
mixed = Counter()
#for i, doc in enumerate(documents("/home/nevidomy/data/ruvimx/UkrLM-wiki/wikipedia/train-00000-of-00012.parquet")):
#for i, doc in enumerate(documents("/home/nevidomy/data/fineweb-2/data/ukr_Cyrl/train/000_00000.parquet")):
#    print("=" * 60)
#    print(f"doc {i}  |  {len(doc)} chars")
#    print(clean_ua_str(doc[:100]))
#    docs.append(doc)
#    if i == 1000: break

get_dataset_stats("/home/nevidomy/data/fineweb-2/data/ukr_Cyrl/train/000_00000.parquet")
get_dataset_stats("/home/nevidomy/data/ruvimx/UkrLM-wiki/wikipedia/train-*-of-00012.parquet")


Pattern: /home/nevidomy/data/fineweb-2/data/ukr_Cyrl/train/000_00000.parquet [1 files]
2,575,000 docs | 8,150,079,460 chars | 1.81 bytes/char
Pattern: /home/nevidomy/data/ruvimx/UkrLM-wiki/wikipedia/train-*-of-00012.parquet [12 files]
1,134,416 docs | 3,178,950,092 chars | 1.74 bytes/char


In [6]:
"""Tokenizer cell -- everything prefixed tk_ / TK_ for a flat notebook namespace.

Assumes clean_ua_str(s) is already defined (it calls scrub internally).
"""

import contextlib, json, os, sys, threading
from tokenizers import Tokenizer, Regex, models, pre_tokenizers, decoders
from tokenizers import trainers, processors
import random

TK_BOS, TK_EOS = "<|bos|>", "<|eos|>"
TK_VOCAB = 32768          # 2**15 total, specials and 256-byte alphabet included
TK_MAX_TOKEN_LEN = 32     # byte-level units; Cyrillic is 2/char, so ~16 chars

# Branches: 1) optional non-letter prefix + letter run, apostrophes kept
#           2) a bare single digit -- no prefix, so ids are context-free
#           3) punctuation run absorbing trailing newlines
#           4) newline runs   5) remaining whitespace
TK_PATTERN = Regex(
    r"[^\r\n\p{L}\p{N}]?[\p{L}\p{M}]+(?:'[\p{L}\p{M}]+)*"
    r"|\p{N}"
    r"|[^\s\p{L}\p{N}]+[\r\n]*"
    r"|\s*[\r\n]+"
    r"|\s+"
)


def tk_build() -> Tokenizer:
    tok = Tokenizer(models.BPE())
    tok.pre_tokenizer = pre_tokenizers.Sequence([
        pre_tokenizers.Split(TK_PATTERN, behavior="isolated"),
        pre_tokenizers.ByteLevel(add_prefix_space=False, use_regex=False),
    ])
    tok.decoder = decoders.ByteLevel()
    return tok


def tk_finalize(tok: Tokenizer) -> Tokenizer:
    bos, eos = tok.token_to_id(TK_BOS), tok.token_to_id(TK_EOS)
    tok.post_processor = processors.Sequence([
        processors.ByteLevel(trim_offsets=False),
        processors.TemplateProcessing(
            single=f"{TK_BOS} $A {TK_EOS}",
            special_tokens=[(TK_BOS, bos), (TK_EOS, eos)],
        ),
    ])
    return tok


# ---------------------------------------------------------------- progress --
@contextlib.contextmanager
def tk_progress(enabled=True):
    """indicatif output vanishes on a non-TTY (notebooks included); the json
    format always emits, so capture fd 2 and drive tqdm from it."""
    try:
        from tqdm.auto import tqdm
    except ImportError:
        tqdm = None
    if not enabled or tqdm is None:
        yield
        return

    read_fd, write_fd = os.pipe()
    saved = os.dup(2)
    os.dup2(write_fd, 2)
    os.close(write_fd)
    bars, other = {}, []

    def pump():
        with os.fdopen(read_fd, "r", buffering=1, errors="replace") as pipe:
            for line in pipe:
                try:
                    ev = json.loads(line)
                    stage, cur, tot = ev["stage"], ev["current"], ev["total"]
                except (ValueError, KeyError):
                    other.append(line)
                    continue
                bar = bars.get(stage)
                if bar is None:
                    bar = bars[stage] = tqdm(total=tot, desc=stage,
                                             leave=True, file=sys.stdout)
                if tot != bar.total:
                    bar.total = tot
                    bar.refresh()
                bar.update(cur - bar.n)

    t = threading.Thread(target=pump, daemon=True)
    t.start()
    try:
        yield
    finally:
        os.dup2(saved, 2)
        os.close(saved)
        t.join(timeout=5)
        for b in bars.values():
            b.close()
        for line in other:
            sys.stderr.write(line)


# ------------------------------------------------------------------ sample --
def tk_write_sample(doc_iter, out_path, target_chars=1_500_000_000,
                    clean_fn=None, workers=None):
    """Clean documents into a JSONL file. JSONL, not one-per-line text, so the
    \\n\\n paragraph breaks survive. Returns the document count."""
    from tqdm.auto import tqdm
    clean_fn = clean_fn or clean_ua_str            # noqa: F821
    total = n = 0

    def _emit(f, out):
        f.write(json.dumps(out, ensure_ascii=False) + "\n")

    with open(out_path, "w", encoding="utf-8") as f:
        if workers == 1:
            it = (clean_fn(d) for d in doc_iter)
        else:
            from multiprocessing import Pool
            pool = Pool(workers)
            it = pool.imap_unordered(clean_fn, doc_iter, chunksize=200)
        for out in tqdm(it, unit="doc", file=sys.stdout):
            if not out:
                continue
            _emit(f, out)
            n += 1
            total += len(out)
            if total >= target_chars:
                break
        if workers != 1:
            pool.terminate()
    print(f"{n:,} docs, {total/1e6:.1f}M chars -> {out_path}")
    return n


def tk_read_sample(path):
    """json.loads is C-level, so this feeds the Rust trainer at I/O speed."""
    with open(path, encoding="utf-8") as f:
        for line in f:
            yield json.loads(line)


# ------------------------------------------------------------------- train --
def tk_train(corpus_iter, vocab_size=TK_VOCAB, min_frequency=2,
             max_token_length=TK_MAX_TOKEN_LEN, length=None, progress=True):
    tok = tk_build()
    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size,
        min_frequency=min_frequency,
        max_token_length=max_token_length,
        special_tokens=[TK_BOS, TK_EOS],
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
        show_progress=progress,
        progress_format="json" if progress else "none",
    )
    with tk_progress(progress):
        tok.train_from_iterator(corpus_iter, trainer=trainer, length=length)
    return tk_finalize(tok)


def tk_fertility(tok, docs):
    encs = tok.encode_batch(docs, add_special_tokens=False)
    nch = sum(len(d) for d in docs)
    ntk = sum(len(e.ids) for e in encs)
    return nch / ntk, ntk / len(docs)


def tk_check(tok):
    """Invariants worth asserting before spending GPU time."""
    assert tok.get_vocab_size() < 2**16, "uint16 memmap would wrap"
    ids = {}
    for ctx in ["{d}", " {d}", "-{d}", ".{d}", "рік {d} рік", "({d})"]:
        for d in "0123456789":
            e = tok.encode(ctx.format(d=d), add_special_tokens=False)
            pos = [i for i, t in enumerate(e.tokens) if t == d]
            assert len(pos) == 1, f"{ctx.format(d=d)!r} -> {e.tokens}"
            ids.setdefault(d, e.ids[pos[0]])
            assert ids[d] == e.ids[pos[0]], f"digit {d} unstable in {ctx!r}"
    for s in ["п'ять об'єктів", "Абзац\n\nДругий.", "Київ — столиця."]:
        assert tok.decode(tok.encode(s).ids, skip_special_tokens=True) == s, s
    e = tok.encode("Київ")
    assert e.tokens[0] == TK_BOS and e.tokens[-1] == TK_EOS
    nl = [t for t in tok.get_vocab() if "Ċ" in t]
    print(f"vocab {tok.get_vocab_size()}, digits stable, "
          f"roundtrip ok, {len(nl)} newline-bearing tokens")

import contextlib
import json
import sys


def tk_write_sample(doc_iter, out_path, target_chars=None,
                    clean_fn=None, workers=None):
    """Clean documents into a JSONL file.

    JSONL rather than one-doc-per-line text, so \\n\\n paragraph breaks survive.

    target_chars=None means "consume the whole iterator" -- use that when
    tk_sample_sources already owns the per-source budgets. Passing a number
    here imposes a second, global cap on top of them.

    Returns the document count, for tk_train(..., length=n).
    """
    from tqdm.auto import tqdm
    clean_fn = clean_fn or clean_ua_str            # noqa: F821
    total = n = 0

    with contextlib.ExitStack() as stack:
        f = stack.enter_context(open(out_path, "w", encoding="utf-8"))
        if workers == 1:
            it = (clean_fn(d) for d in doc_iter)
        else:
            from multiprocessing import Pool
            pool = stack.enter_context(Pool(workers))   # __exit__ terminates
            it = pool.imap_unordered(clean_fn, doc_iter, chunksize=64)
        bar = stack.enter_context(tqdm(unit="doc", file=sys.stdout))

        for out in it:
            if not out:
                continue
            f.write(json.dumps(out, ensure_ascii=False) + "\n")
            n += 1
            total += len(out)
            bar.update(1)
            if target_chars is not None and total >= target_chars:
                break

    print(f"{n:,} docs, {total/1e6:.2f}M chars -> {out_path}")
    return n

def tk_sample_sources(sources, seed=0, column="text", verbose=True):
    """Sample raw documents from several Parquet sources, each with its own
    character budget.

    sources: list of (glob_pattern, target_chars)

    Yields RAW text -- cleaning belongs in the worker pool downstream, not in
    this generator, or the whole pipeline serialises on one core.

    Row groups are drawn without replacement, so no document is emitted twice.
    Budgets count raw characters; cleaning shrinks that a few percent.
    """
    rng = random.Random(seed)
    for pattern, target in sources:
        shards = sorted(glob.glob(pattern))
        if not shards:
            raise FileNotFoundError(f"no files match {pattern!r}")

        handles = {p: pq.ParquetFile(p) for p in shards}
        units = [(p, i) for p in shards
                 for i in range(handles[p].metadata.num_row_groups)]
        rng.shuffle(units)

        total = ndocs = 0
        done = False
        for path, rg in units:
            if done:
                break
            for b in handles[path].iter_batches(batch_size=2000,
                                                columns=[column],
                                                row_groups=[rg]):
                for d in b.column(column).to_pylist():
                    if not d:
                        continue
                    total += len(d)
                    ndocs += 1
                    yield d
                    if total >= target:    # per-document, so small budgets
                        done = True        # aren't blown by one row group
                        break
                if done:
                    break

        if verbose:
            pct = total / target * 100
            note = "" if total >= target else "  <-- EXHAUSTED, under budget"
            print(f"{pattern}: {ndocs:,} docs, {total/1e6:.2f}M raw chars "
                  f"({pct:.0f}% of budget){note}")

In [32]:
SOURCES = [
    ("/home/nevidomy/data/fineweb-2/data/ukr_Cyrl/train/*.parquet", 1_200_000_000),
    ("/home/nevidomy/data/ruvimx/UkrLM-wiki/wikipedia/train-*-of-00012.parquet", 300_000_000),
]
#n = tk_write_sample(tk_sample_sources(SOURCES), "sample.jsonl", target_chars=None)
n = 4927892
tok = tk_train(tk_read_sample("sample.jsonl"), length=n, vocab_size=3*2**14)
tok.save("ua_bpe_48k.json")

held = [clean_ua_str(d) for d in tk_sample_sources(
   [("/home/nevidomy/data/fineweb-2/data/ukr_Cyrl/train/000_00000.parquet", 3_000),
    ("/home/nevidomy/data/ruvimx/UkrLM-wiki/wikipedia/train-*-of-00012.parquet", 2_000)],
   verbose=False)]

encs = tok.encode_batch(held, add_special_tokens=False)
cpt = sum(len(s) for s in held) / sum(len(e.ids) for e in encs)
print(f"{cpt:.2f} chars/token")



Tokenize words: 100%|██████████| 4927892/4927892 [00:08<00:00, 611295.68it/s]




Tokenize words: 100%|██████████| 4927892/4927892 [00:24<00:00, 611295.68it/s]


























































































































































































































































































































































































































































































































































































































































Compute merges: 100%|██████████| 48894/48894 [01:58<00:00, 412.82it/s] 
3.80 chars/token


In [24]:
"""Byte-level BPE vocab inspection -- self-contained, tk_ prefixed.

Byte-level BPE stores tokens in GPT-2's printable-byte alphabet so the vocab
file stays JSON-safe: Ġ is a space (0x20), Ċ a newline (0x0A), and Cyrillic
shows as two characters each because it is 2 bytes in UTF-8. This decodes it
back and flags what is worth looking at.
"""

import json
from collections import Counter


def _tk_byte_decoder():
    """Inverse of GPT-2's bytes_to_unicode."""
    bs = (list(range(ord("!"), ord("~") + 1))
          + list(range(ord("\u00a1"), ord("\u00ac") + 1))
          + list(range(ord("\u00ae"), ord("\u00ff") + 1)))
    cs = bs[:]
    n = 0
    for b in range(256):
        if b not in bs:
            bs.append(b)
            cs.append(256 + n)
            n += 1
    return {chr(c): b for b, c in zip(bs, cs)}


_TK_DEC = _tk_byte_decoder()


def tk_bytes(token: str) -> bytes:
    try:
        return bytes(_TK_DEC[c] for c in token)
    except KeyError:                       # special token like <|bos|>
        return token.encode("utf-8")


def tk_show(token: str) -> str:
    """'ĠÑģÑĤÐ¾Ð»' -> ' стол'. Partial tokens straddle a character boundary,
    which is normal for byte-level BPE on a non-Latin script."""
    raw = tk_bytes(token)
    try:
        return raw.decode("utf-8")
    except UnicodeDecodeError:
        return f"<partial {raw.hex()}>"


def tk_is_partial(token: str) -> bool:
    try:
        tk_bytes(token).decode("utf-8")
        return False
    except UnicodeDecodeError:
        return True


def _tk_vocab(src):
    """Accepts a Tokenizer, a path to a saved json, or a vocab dict."""
    if isinstance(src, dict):
        return src
    if isinstance(src, str):
        return json.load(open(src, encoding="utf-8"))["model"]["vocab"]
    return src.get_vocab()


def tk_audit(src, sample=15):
    vocab = _tk_vocab(src)
    inv = {i: t for t, i in vocab.items()}

    kinds = Counter()
    for t in inv.values():
        d = tk_show(t)
        if d.startswith("<partial "):
            kinds["partial utf-8 (straddles a char)"] += 1
        elif any("\u0400" <= c <= "\u04FF" for c in d):
            kinds["Cyrillic"] += 1
        elif any(c.isascii() and c.isalpha() for c in d):
            kinds["Latin"] += 1
        elif any(c.isdigit() for c in d):
            kinds["digit"] += 1
        else:
            kinds["punctuation/space"] += 1

    print(f"vocab size: {len(vocab):,}")
    for k, v in kinds.most_common():
        print(f"  {k:34} {v:>7,}  {v / len(vocab) * 100:5.1f}%")

    # sort by BYTE length; the "<partial ...>" placeholder is not the token
    whole = sorted(((i, t) for i, t in inv.items() if not tk_is_partial(t)),
                   key=lambda kv: -len(tk_bytes(kv[1])))
    print(f"\nlongest {sample} complete tokens:")
    for i, t in whole[:sample]:
        print(f"  {i:>6}  {len(tk_bytes(t)):>3}B  {tk_show(t)!r}")

    part = sorted(((i, t) for i, t in inv.items() if tk_is_partial(t)),
                  key=lambda kv: -len(tk_bytes(kv[1])))
    if part:
        print(f"\nlongest {min(sample, len(part))} partial tokens:")
        for i, t in part[:sample]:
            print(f"  {i:>6}  {len(tk_bytes(t)):>3}B  {tk_show(t)}")
    return inv


def tk_partial_usage(tok, docs):
    """Share of EMITTED tokens that straddle a character boundary. Vocab share
    is not the number that matters -- this one is."""
    partial_ids = {i for t, i in tok.get_vocab().items() if tk_is_partial(t)}
    n = bad = 0
    for e in tok.encode_batch(docs, add_special_tokens=False):
        n += len(e.ids)
        bad += sum(1 for i in e.ids if i in partial_ids)
    print(f"{bad:,} / {n:,} emitted tokens are partial  ({bad / n * 100:.2f}%)")
    return bad / n


tk_audit("ua_bpe_32k.json")
tk_partial_usage(tok, held)

vocab size: 32,768
  Cyrillic                            30,371   92.7%
  Latin                                1,916    5.8%
  punctuation/space                      310    0.9%
  partial utf-8 (straddles a char)       159    0.5%
  digit                                   12    0.0%

longest 15 complete tokens:
    2598   31B  ' використовувати'
    9721   31B  ' спостерігається'
   11176   31B  ' використовували'
   11626   31B  ' співробітництва'
   12354   31B  ' адміністративно'
   13281   31B  ' транспортування'
   13311   31B  ' Всеукраїнського'
   14007   31B  ' життєдіяльності'
   14459   31B  ' супроводжується'
   16349   31B  ' інтелектуальної'
   16889   31B  ' індивідуального'
   17312   31B  ' підприємницької'
   17542   31B  ' бухгалтерського'
   17631   31B  ' адміністративні'
   20226   31B  ' спеціалізованих'

longest 15 partial tokens:
     267    3B  <partial d0bed0>
     258    2B  <partial 20d0>
     350    2B  <partial e280>
    1222    2B  <partial e284>
    3463

0.0

In [1]:
"""Script census -- find documents that are not actually Ukrainian.
 
The partial tokens in a byte-level vocab act as a free contamination alarm:
BPE only spends a merge on a byte pair that recurs, so a vocab slot for
hiragana lead bytes means there is real Japanese text in the corpus. This
measures how much, and which documents carry it.
"""
 
import sys
import unicodedata
from collections import Counter
 
_RANGES = [
    ("cyrillic", 0x0400, 0x052F),
    ("latin",    0x0041, 0x024F),
    ("greek",    0x0370, 0x03FF),
    ("hebrew",   0x0590, 0x05FF),
    ("arabic",   0x0600, 0x06FF),
    ("kana",     0x3040, 0x30FF),
    ("cjk",      0x3400, 0x9FFF),
    ("hangul",   0xAC00, 0xD7AF),
    ("emoji",    0x1F000, 0x1FAFF),
]
 
 
def tk_script(ch):
    cp = ord(ch)
    for name, lo, hi in _RANGES:
        if lo <= cp <= hi:
            return name
    cat = unicodedata.category(ch)
    if cat.startswith("N"):
        return "digit"
    if cat[0] in "ZCP" or ch.isspace():
        return "punct/space"
    return "other"
 
 
def tk_doc_profile(doc):
    """Per-document script shares, computed over letters only -- punctuation
    and digits are script-neutral and would dilute the signal."""
    c = Counter(tk_script(ch) for ch in doc)
    letters = sum(v for k, v in c.items() if k not in ("digit", "punct/space"))
    return c, letters
 
 
def tk_script_census(docs, limit=100_000, cyr_floor=0.5, show=10):
    totals = Counter()
    suspect = []
    n = 0
    for doc in docs:
        if n >= limit:
            break
        n += 1
        c, letters = tk_doc_profile(doc)
        totals.update(c)
        if letters < 50:
            continue
        cyr = c.get("cyrillic", 0) / letters
        if cyr < cyr_floor:
            dominant = max(((k, v) for k, v in c.items()
                            if k not in ("digit", "punct/space", "cyrillic")),
                           key=lambda kv: kv[1], default=("?", 0))
            suspect.append((cyr, dominant[0], len(doc), doc[:160]))
 
    grand = sum(totals.values())
    print(f"{n:,} docs, {grand/1e6:.1f}M chars\n")
    for k, v in totals.most_common():
        print(f"  {k:12} {v:>14,}  {v/grand*100:7.3f}%")
 
    print(f"\n{len(suspect):,} docs below {cyr_floor:.0%} Cyrillic "
          f"({len(suspect)/n*100:.2f}% of sample)")
    by_script = Counter(s[1] for s in suspect)
    for k, v in by_script.most_common():
        print(f"  dominant {k:10} {v:>7,} docs")
 
    suspect.sort()
    print(f"\nworst {min(show, len(suspect))}:")
    for cyr, dom, ln, head in suspect[:show]:
        print(f"  cyr={cyr:5.1%} {dom:8} {ln:>7}ch  {head!r}")
    return suspect
 
 
if __name__ == "__main__" and False:
    docs = [
        "Київ є столицею України та найбільшим містом країни за населенням.",
        "システムは日本語のテキストです。これはウクライナ語ではありません。" * 3,
        "Водень — це перший елемент періодичної таблиці Менделєєва у 1869 році.",
        "This document is entirely in English and should be flagged too here.",
        "Ціна становить 250 ₴, номер № 42, температура ≈ 20 °C — усе гаразд.",
        "Σύστημα ελληνικού κειμένου που δεν είναι ουκρανικό κείμενο καθόλου.",
    ] * 40
    s = tk_script_census(docs, show=4)
    assert any(d[1] == "kana" for d in s), "kana not detected"
    assert any(d[1] == "latin" for d in s), "latin not detected"
    print("\ndetector works")
 
def tk_usage(tok, docs, cuts=(1024, 2048, 4096, 8192, 16384, 24576, 32768)):
    counts = Counter()
    total = 0
    for e in tok.encode_batch(docs, add_special_tokens=False):
        counts.update(e.ids)
        total += len(e.ids)
 
    V = tok.get_vocab_size()
    used = len(counts)
    print(f"{total:,} tokens emitted over {len(docs):,} docs")
    print(f"{used:,} / {V:,} slots ever used  ({used/V*100:.1f}%)")
    for thresh in (1, 10, 100):
        n = sum(1 for c in counts.values() if c <= thresh)
        print(f"  {n:>7,} slots used <= {thresh:>3}x  ({n/V*100:5.1f}%)")
 
    # coverage by id order: ids are assigned in merge order, so a prefix of the
    # id space is exactly the vocab you would have got at that smaller size
    print("\nshare of emitted tokens whose id falls below N "
          "(~ what a size-N vocab would have covered directly):")
    for n in cuts:
        if n > V:
            break
        share = sum(c for i, c in counts.items() if i < n) / total
        print(f"  id < {n:>6}  {share*100:6.2f}%")
 
    print("\nleast-used slots that were still learned:")
    rare = sorted(counts.items(), key=lambda kv: kv[1])[:10]
    inv = {i: t for t, i in tok.get_vocab().items()}
    for i, c in rare:
        print(f"  id {i:>6}  seen {c:>4}x  {inv.get(i)!r}")
    return counts
 
 
def tk_compare_vocabs(paths, docs):
    """paths: {label: tokenizer_path}. Fertility on identical held-out text."""
    from tokenizers import Tokenizer
    print(f"{'vocab':>8}  {'chars/token':>12}  {'tokens/doc':>11}  "
          f"{'emb params @768':>16}")
    nch = sum(len(d) for d in docs)
    base = None
    for label, path in paths.items():
        t = Tokenizer.from_file(path)
        ntk = sum(len(e.ids) for e in
                  t.encode_batch(docs, add_special_tokens=False))
        cpt = nch / ntk
        emb = t.get_vocab_size() * 768
        delta = "" if base is None else f"   {(cpt/base - 1)*100:+.2f}% vs first"
        base = base or cpt
        print(f"{t.get_vocab_size():>8,}  {cpt:>12.3f}  {ntk/len(docs):>11.0f}  "
              f"{emb/1e6:>13.1f}M{delta}")


#suspect = tk_script_census(tk_sample_sources(SOURCES, seed=7, verbose=False), limit=200_000)

tk_usage(tok, held)


NameError: name 'tok' is not defined

In [ ]:
"""What would a model that learned NOTHING but token statistics achieve?
 
Three floors, each computable directly from train.bin:
 
  ln(V)              a uniform model -- your loss at init
  unigram entropy    a model that learned only token frequencies
  bigram entropy     a model that learned only the previous token
 
A transformer that has not beaten the bigram floor has not yet learned
anything a lookup table could not do. These are exact, not estimates, so they
calibrate 'is 5.53 good' without needing a reference model.
"""
 
import json
import os
 
import numpy as np
 
 
def tk_baselines(out_dir, split="train", eval_split="val", vocab_size=None,
                 chars_per_token=None):
    meta_path = os.path.join(out_dir, "meta.json")
    if vocab_size is None and os.path.exists(meta_path):
        vocab_size = json.load(open(meta_path))["vocab_size"]
 
    tr = np.memmap(os.path.join(out_dir, f"{split}.bin"), dtype=np.uint16,
                   mode="r")
    va_path = os.path.join(out_dir, f"{eval_split}.bin")
    va = np.memmap(va_path, dtype=np.uint16, mode="r") if os.path.exists(va_path) else tr
 
    V = int(vocab_size or max(tr.max(), va.max()) + 1)
    rows = []
 
    rows.append(("uniform  ln(V)", float(np.log(V))))
 
    # ---- unigram, fit on train, evaluated on val -------------------------
    cnt = np.bincount(np.asarray(tr, dtype=np.int64), minlength=V).astype(np.float64)
    p = (cnt + 1.0) / (cnt.sum() + V)                     # Laplace, avoids inf
    nll = -np.log(p)[np.asarray(va, dtype=np.int64)]
    rows.append(("unigram", float(nll.mean())))
 
    # ---- bigram ----------------------------------------------------------
    if V * V <= 200_000_000:
        prev = np.asarray(tr[:-1], dtype=np.int64)
        nxt = np.asarray(tr[1:], dtype=np.int64)
        joint = np.bincount(prev * V + nxt, minlength=V * V).astype(np.float32)
        joint = joint.reshape(V, V)
        joint += 1.0                                      # Laplace
        cond = joint / joint.sum(axis=1, keepdims=True)
        vp = np.asarray(va[:-1], dtype=np.int64)
        vn = np.asarray(va[1:], dtype=np.int64)
        b = -np.log(cond[vp, vn].astype(np.float64))
        rows.append(("bigram", float(b.mean())))
        del joint, cond
    else:
        rows.append(("bigram", float("nan")))
 
    print(f"vocab {V:,} | train {len(tr):,} tok | eval {len(va):,} tok\n")
    hdr = f"{'floor':<16}{'nats/token':>12}"
    if chars_per_token:
        hdr += f"{'nats/char':>12}{'bits/char':>12}"
    print(hdr)
    for name, v in rows:
        line = f"{name:<16}{v:>12.4f}"
        if chars_per_token:
            npc = v / chars_per_token
            line += f"{npc:>12.4f}{npc/np.log(2):>12.4f}"
        print(line)
    return dict(rows)
 
 
def tk_verdict(loss, floors, chars_per_token=None):
    print(f"\nmodel loss {loss:.4f} nats/token", end="")
    if chars_per_token:
        npc = loss / chars_per_token
        print(f"  =  {npc:.4f} nats/char  =  {npc/np.log(2):.4f} bits/char")
    else:
        print()
    for name, v in floors.items():
        if np.isnan(v):
            continue
        gap = v - loss
        verdict = "BEATS" if gap > 0 else "worse than"
        print(f"  {verdict:>10} {name:<16} by {abs(gap):6.4f} nats")
 